# Notebook 19 — E4 five-seed reproducibility & two fair 444-feature context controls

**This experiment has NOT produced results yet.** It tests the recorded E3 breakthrough (84.5079% accuracy, 64.7510% ten-class macro-F1) with five algorithm seeds (42–46) on **one original fixed 35,005-window internal validation partition**. This is a replicability screen, not five independently sampled test sets.

## Preregistered conditions

- **Intact:** Exactly E3's rare-weighted LightGBM on 444 features (five CSV-order records × 42 + 234 within-window numerical statistics). **Seed42 original fitted checkpoint restored and validated, never trained again**; only seeds 43–46 are trained.
- **Block-shuffled:** All four historical records (the 4 × 42 values) from another window transferred together, preserving relationships among historical positions but breaking alignment with target record; statistics recomputed from disrupted context. Five model seeds.
- **Independently shuffled:** Each of four historical records is permuted separately as a coherent 42-feature record, preserving within-record validity but breaking target–history and inter-history alignment; statistics recomputed. Five model seeds.

For both controls the 42 target-record features and labels are untouched. Train and validation partitions are permuted separately with fixed distinct seeds. All three variants have the same **444-feature** model input, identical class weighting and LightGBM hyperparameters, and exactly the same 35,005 validation target labels.

**Critically** the UNSW TRAINING CSV row order has not been verified as true chronological order. This experiment tests dependence on serialization alignment, not evidence of physical temporal traffic causality.

## Crash recovery and provenance

Each completely trained seed candidate receives its own true confusion matrix, per-class F1/recall, probabilities, fitted model checkpoint and training history. Each candidate is **immediately SHA256-verified and uploaded to an unpublished GitHub draft release**; public commits contain only aggregate metrics/provenance. A fresh Kaggle run will restore previously archived candidate checkpoints rather than retrain them. The two frozen E2 models, seed42 E3 model, original SCV split and 42-feature controls are also restored from pinned private releases.

**Kaggle**: attach `dsshayan/unsw-nb15`, Internet **ON**, authorize the existing `pushRare` secret, Accelerator **None / CPU**. Start via **Save Version → Save & Run All** to run independently of your browser. The total job is 14 new CPU model fits with progress bars and private backups after each fit. If the job exceeds Kaggle's execution allowance, completed seed backups survive.

**Caveat:** Early stopping uses our same internal validation split, so no final superiority claim is possible. Run independent train/test evaluation only after the methodology is frozen.

Final marker: `WINDOW-E4-REPL-001 FIVE-SEED RESULTS VERIFIED AND ARCHIVED`.

In [ ]:
# ONE CELL: restore frozen E2/E3, fit 14 new E4 CPU models, archive each seed.
import os,sys,subprocess,hashlib
from pathlib import Path
from kaggle_secrets import UserSecretsClient

REPO=Path("/kaggle/working/rare-attack-ids-window-e4")
BASE=Path("/kaggle/working/research_outputs")
E2=BASE/"WINDOW-E2-001-S42"
E3=BASE/"WINDOW-E3-001-S42"
OUT=BASE/"WINDOW-E4-REPL-001"
CSV=Path("/kaggle/input/datasets/dsshayan/unsw-nb15/UNSW_NB15_training-set.csv")
assert CSV.is_file(), "Attach dsshayan/unsw-nb15 dataset"
with CSV.open("rb") as fh:
    assert hashlib.file_digest(fh,"sha256").hexdigest()==(
        "bec7dd5ec88dc2a0ccc7a07879d338395ed7421750f675fd0339e07dfe0648fa"
    ),"Frozen training input checksum differs!"
secret=UserSecretsClient().get_secret("pushRare")
assert secret, "Enable existing Kaggle pushRare secret in Notebook 19"
if REPO.is_dir():
    subprocess.run(["git","-C",str(REPO),"pull","--ff-only"],check=True)
else:
    subprocess.run(["git","clone","--depth","1",
        "https://github.com/itsCodeBakery/rare-attack-spatiotemporal-ids.git",
        str(REPO)],check=True)
env=os.environ.copy()
env["GITHUB_TOKEN"]=secret
env["PYTHONPATH"]=str(REPO/"src")
env["OMP_NUM_THREADS"]="4"
env["OPENBLAS_NUM_THREADS"]="1"
print("E4 EXPERIMENT CODE COMMIT:",
    subprocess.check_output(["git","-C",str(REPO),"rev-parse","HEAD"],text=True).strip(),
    flush=True)

# No previous models are retrained.
for name,archive in [
    ("restore_scv_release.py","SCV-IDS-REPRO-001_FULL_ARTIFACTS.tar"),
    ("restore_ctrl003_tabular.py","CTRL-TAB-001-S42_FULL_ARTIFACTS.tar")]:
    subprocess.run([
        sys.executable,str(REPO/"scripts"/name),
        "--base",str(BASE),"--archive",str(Path("/kaggle/working")/archive)],
        cwd=str(REPO),env=env,check=True)

sys.path.insert(0,str(REPO/"scripts"))
from archive_window_e2 import restore_prior_candidate as restore_e2,validate as verify_e2
from archive_window_e3 import restore_prior_candidate as restore_e3,validate as verify_e3
for name in ("window_unweighted","window_rare_sqrt"):
    restore_e2(E2,name,secret)
    verify_e2(E2,name)
    print("Restored original E2 control:",name,flush=True)
restore_e3(E3,"stats_rare_sqrt",secret)
verify_e3(E3,"stats_rare_sqrt")
print("Restored actual E3 seed42 fitted model; NO seed42 retraining",flush=True)

subprocess.run([
    sys.executable,str(REPO/"scripts/run_window_e4.py"),
    "--repo",str(REPO),
    "--reference",str(BASE/"SCV-IDS-REPRO-001"),
    "--baseline",str(BASE/"CTRL-TAB-001-S42"),
    "--e2",str(E2),"--e3",str(E3),
    "--config",str(REPO/"configs/window_e4_replication_001.json"),
    "--output",str(OUT),"--auto-archive"
],cwd=str(REPO),env=env,check=True)
print("REAL FIVE-SEED SUMMARY:")
print((OUT/"five_seed_summary.csv").read_text(),flush=True)
print("REAL PAIRED SHUFFLE CONTROLS:")
print((OUT/"paired_differences.csv").read_text(),flush=True)
print("ALL FITTED MODEL WEIGHTS AND PRIVATE PROBABILITIES SECURELY BACKED UP",flush=True)
